# Transform Races Data

1. Read bronze `races` table
2. Keep only the columns required for Analytics (drop url column)
3. Standardise column names using snake_case
4. Rename columns to make them more meaningful
5. Filter out rows where `season` and `round` is null
6. Remove duplicated values
7. Transform values of columns in Title Case
8. Write the transformted data to a silver table 

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.races'
silver_table = f'{catalog_name}.{silver_schema}.races'

## Step 1 - Read bronze races table

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
display(races_df)

## Step 2 - Keep only the columns required for Analytics


In [0]:
from pyspark.sql import functions as F

In [0]:
races_df_selected = races_df.select(
    F.col('season'),
    F.col('round'),
    F.col('raceName'),
    F.col('date'),
    F.col('circuitId'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)

## Step 3 and 4 - Standardise column names using snake_case

In [0]:
races_df_renamed = races_df_selected.withColumnsRenamed(
    {
        'raceName': 'race_name',
        'circuitId': 'circuit_id'
    }
)

## Step 5 - Filter out rows where `season` and `round` is null


In [0]:
races_df_filtered = races_df_renamed.filter((F.col('season').isNotNull() & F.col('round').isNotNull()))

## Step 6 - Remove duplicated values


In [0]:
races_df_distinct = races_df_filtered.distinct()

## Step 7 - Transform values of columns in Title Case

In [0]:
races_df_final = races_df_distinct.withColumn('race_name', F.initcap('race_name'))

## Step 8 - Write the transformted data to a silver table 

In [0]:
(
    races_df_final.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
SELECT * FROM formula1.silver.races
